   # 04 — Final Functional Classification



   This notebook builds the final, thesis-ready comparison across the

   functional and interpretable models that are relevant for the project.



   It does four things:

   1. loads the sampled Gaia XP spectra as functional data

   2. reruns the selected distance-based models on the shared repeated CV splits

   3. reruns functional linear models directly on the spectra

   4. imports the FPCA / FPLS summary and combines all final results



   Main model families:

   - distance-based functional classifiers

   - FPCA / FPLS classifiers

   - functional linear classifiers



   Main evaluation metrics:

   - PR-AUC

   - ROC-AUC

   - Sensitivity

   - Precision

   - Specificity

   - Accuracy

   - F1

   - Youden J



   Main exported outputs:

   - final_distance_fold_metrics.csv

   - final_linear_fold_metrics.csv

   - final_model_summary.csv

   - final_rank_by_f1.csv

   - final_best_by_family.csv

   - interpretability_payload.npz

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


   ## 1. Configuration

In [ ]:
BASE_DIR = Path.cwd() / "og_data"
OUT_DIR = Path.cwd() / "results" / "04_final_functional_models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SMOKE = False

# Distance-based models
DISTANCE_K = 5
DISTANCE_MODELS = [
    {"model_name": "weighted kNN", "distance": "euclidean", "weighted": True},
    {"model_name": "weighted kNN", "distance": "cosine", "weighted": True},
    {"model_name": "weighted kNN", "distance": "seuclidean", "weighted": True},
    {"model_name": "weighted kNN", "distance": "correlation", "weighted": True},
    {"model_name": "centroid", "distance": "euclidean", "weighted": False},
    {"model_name": "centroid", "distance": "cosine", "weighted": False},
    {"model_name": "centroid", "distance": "seuclidean", "weighted": False},
    {"model_name": "centroid", "distance": "correlation", "weighted": False},
]

# Functional linear models
LINEAR_MODELS = [
    {
        "family": "Functional linear",
        "method": "Functional logistic regression (L2)",
        "kind": "logreg_l2",
    },
    {
        "family": "Functional linear",
        "method": "Functional logistic regression (L1)",
        "kind": "logreg_l1",
    },
    {
        "family": "Functional linear",
        "method": "Functional linear SVM",
        "kind": "linear_svm",
    },
]

# Practical speed-up:
# smaller inner CV and smaller C-grid than before
INNER_CV = 3
CS_GRID = np.logspace(-2, 2, 6)

# Imported summary from the FPCA/FPLS notebook
FPCA_FPLS_SUMMARY_FILE = OUT_DIR / "fpca_fpls_summary.csv"


   ## 2. File helpers

In [ ]:
def find_first_existing(paths: list[Path]) -> Path | None:
    for p in paths:
        if p.exists():
            return p
    return None


def split_sort_key(k: str) -> tuple[int, int]:
    rep = int(k.split("_")[0].replace("rep", ""))
    fold = int(k.split("_")[1].replace("fold", ""))
    return (rep, fold)


def ms(mean, std):
    if pd.isna(mean):
        return np.nan
    return f"{mean:.4f} ± {0.0 if pd.isna(std) else std:.4f}"


def normalize_scores_train_ref(scores_te: np.ndarray, scores_tr: np.ndarray) -> np.ndarray:
    lo = float(np.min(scores_tr))
    hi = float(np.max(scores_tr))
    if hi == lo:
        return np.full_like(scores_te, 0.5, dtype=np.float64)
    out = (scores_te - lo) / (hi - lo)
    return np.clip(out, 0.0, 1.0).astype(np.float64)


def pick_youden_threshold(y_true: np.ndarray, y_prob: np.ndarray, grid_size: int = 200) -> float:
    thresholds = np.linspace(0, 1, grid_size)
    best_j = -1.0
    best_thr = 0.5

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(np.int64)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        j = sens + spec - 1.0
        if j > best_j:
            best_j = j
            best_thr = float(thr)

    return best_thr


def fold_metrics(
    y_true_te: np.ndarray,
    y_score_te: np.ndarray,
    y_true_tr: np.ndarray,
    y_score_tr: np.ndarray,
) -> dict:
    out = {"pr_auc": average_precision_score(y_true_te, y_score_te)}

    try:
        out["roc_auc"] = float(roc_auc_score(y_true_te, y_score_te))
    except ValueError:
        out["roc_auc"] = np.nan

    prob_tr = normalize_scores_train_ref(y_score_tr, y_score_tr)
    prob_te = normalize_scores_train_ref(y_score_te, y_score_tr)
    thr = pick_youden_threshold(y_true_tr, prob_tr)
    y_pred = (prob_te >= thr).astype(np.int64)

    out["youden_threshold"] = thr
    out["sensitivity"] = recall_score(y_true_te, y_pred, pos_label=1, zero_division=0)
    out["precision"] = precision_score(y_true_te, y_pred, pos_label=1, zero_division=0)
    out["specificity"] = recall_score(y_true_te, y_pred, pos_label=0, zero_division=0)
    out["accuracy"] = accuracy_score(y_true_te, y_pred)
    out["f1"] = f1_score(y_true_te, y_pred, pos_label=1, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true_te, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    out["youden_j"] = sens + spec - 1.0
    return out


def summarise_run(df_run: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    metric_cols = [
        "pr_auc", "roc_auc", "sensitivity", "precision", "specificity",
        "accuracy", "f1", "youden_j", "youden_threshold"
    ]

    agg_dict = {}
    for m in metric_cols:
        agg_dict[f"{m}_mean"] = pd.NamedAgg(column=m, aggfunc="mean")
        agg_dict[f"{m}_std"] = pd.NamedAgg(column=m, aggfunc="std")

    return df_run.groupby(group_cols).agg(**agg_dict).reset_index()


   ## 3. Locate core files

In [ ]:
LABEL_FILE = find_first_existing([BASE_DIR / "og_xp.csv"])
SPLIT_FILE = find_first_existing([BASE_DIR / "splits_rskf.json"])
SAMPLED_FILE = find_first_existing([BASE_DIR / "xp_sampled_spectra.csv"])

if LABEL_FILE is None:
    raise FileNotFoundError("Missing og_xp.csv")
if SPLIT_FILE is None:
    raise FileNotFoundError("Missing splits_rskf.json")
if SAMPLED_FILE is None:
    raise FileNotFoundError("Missing xp_sampled_spectra.csv")

print("LABEL_FILE:", LABEL_FILE)
print("SPLIT_FILE:", SPLIT_FILE)
print("SAMPLED_FILE:", SAMPLED_FILE)
print("FPCA/FPLS summary:", FPCA_FPLS_SUMMARY_FILE if FPCA_FPLS_SUMMARY_FILE.exists() else "not found")


LABEL_FILE: c:\Users\Lenovo\Documents\VDA\Bakalauras\astroflow_project\og_data\og_xp.csv
SPLIT_FILE: c:\Users\Lenovo\Documents\VDA\Bakalauras\astroflow_project\og_data\splits_rskf.json
SAMPLED_FILE: c:\Users\Lenovo\Documents\VDA\Bakalauras\astroflow_project\og_data\xp_sampled_spectra.csv
FPCA/FPLS summary: not found


   ## 4. Load labels, sampled spectra, and repeated splits

In [ ]:
df_labels = pd.read_csv(LABEL_FILE)
if "source_id" not in df_labels.columns or "y" not in df_labels.columns:
    raise ValueError("og_xp.csv must contain source_id and y")

df_spec = pd.read_csv(SAMPLED_FILE)
if "source_id" not in df_spec.columns:
    raise ValueError("xp_sampled_spectra.csv must contain source_id")

wl_cols = [c for c in df_spec.columns if c.startswith("wl_")]
if len(wl_cols) == 0:
    raise ValueError("No sampled spectrum columns found (expected wl_*)")

wavelengths = np.array([float(c.split("_")[1]) for c in wl_cols], dtype=np.float64)

df_m = df_labels[["source_id", "y"]].merge(
    df_spec[["source_id"] + wl_cols],
    on="source_id",
    how="inner",
    validate="one_to_one",
)

X_raw = df_m[wl_cols].to_numpy(dtype=np.float64)
y = df_m["y"].to_numpy(dtype=np.int64)

row_norm = np.linalg.norm(X_raw, axis=1, keepdims=True)
X = np.divide(X_raw, row_norm, out=np.zeros_like(X_raw), where=row_norm > 1e-20)

with open(SPLIT_FILE) as f:
    splits = json.load(f)

split_names = sorted(splits.keys(), key=split_sort_key)
if SMOKE:
    split_names = [k for k in split_names if k.startswith("rep0_")]

print("Merged rows:", len(df_m))
print("X shape:", X.shape)
print("Binary fraction:", round((y == 1).mean(), 4))
print("Number of splits:", len(split_names))
print("Wavelength range:", wavelengths.min(), "to", wavelengths.max())


Merged rows: 2815
X shape: (2815, 343)
Binary fraction: 0.1982
Number of splits: 50
Wavelength range: 336.0 to 1020.0


   ## 5. Distance-model helpers

In [ ]:
def train_distance_params(X_tr: np.ndarray, distance_name: str) -> dict:
    params = {"distance": distance_name}

    if distance_name == "seuclidean":
        V = np.var(X_tr, axis=0, ddof=1)
        V = np.where(V <= 1e-12, 1e-12, V)
        params["V"] = V

    elif distance_name == "correlation":
        params["mu"] = np.mean(X_tr, axis=0)
        params["sigma"] = np.std(X_tr, axis=0) + 1e-12

    return params


def pairwise_distance_matrix(X_a: np.ndarray, X_b: np.ndarray, params: dict) -> np.ndarray:
    dname = params["distance"]

    if dname == "euclidean":
        return pairwise_distances(X_a, X_b, metric="euclidean")

    if dname == "cosine":
        return pairwise_distances(X_a, X_b, metric="cosine")

    if dname == "seuclidean":
        return pairwise_distances(X_a, X_b, metric="seuclidean", V=params["V"])

    if dname == "correlation":
        return pairwise_distances(X_a, X_b, metric="correlation")

    raise ValueError(f"Unsupported distance: {dname}")


def knn_scores_from_distances(
    D: np.ndarray,
    y_ref: np.ndarray,
    k: int,
    weighted: bool = True,
    exclude_self: bool = False,
) -> np.ndarray:
    D = D.copy()

    if exclude_self and D.shape[0] == D.shape[1]:
        np.fill_diagonal(D, np.inf)

    nn_idx = np.argsort(D, axis=1)[:, :k]
    nn_d = np.take_along_axis(D, nn_idx, axis=1)
    nn_y = y_ref[nn_idx]

    if weighted:
        w = 1.0 / np.maximum(nn_d, 1e-12)
        score = (w * nn_y).sum(axis=1) / w.sum(axis=1)
    else:
        score = nn_y.mean(axis=1)

    return score.astype(np.float64)


def centroid_scores_from_distances(
    X_ref: np.ndarray,
    y_ref: np.ndarray,
    X_query: np.ndarray,
    params: dict,
) -> np.ndarray:
    c0 = X_ref[y_ref == 0].mean(axis=0, keepdims=True)
    c1 = X_ref[y_ref == 1].mean(axis=0, keepdims=True)

    d0 = pairwise_distance_matrix(X_query, c0, params).ravel()
    d1 = pairwise_distance_matrix(X_query, c1, params).ravel()

    s0 = 1.0 / np.maximum(d0, 1e-12)
    s1 = 1.0 / np.maximum(d1, 1e-12)
    score = s1 / (s0 + s1)
    return score.astype(np.float64)


def run_distance_model_one_split(
    X: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    model_cfg: dict,
) -> dict:
    X_tr = X[train_idx]
    X_te = X[test_idx]
    y_tr = y[train_idx]
    y_te = y[test_idx]

    params = train_distance_params(X_tr, model_cfg["distance"])

    if model_cfg["model_name"] == "weighted kNN":
        D_tr = pairwise_distance_matrix(X_tr, X_tr, params)
        D_te = pairwise_distance_matrix(X_te, X_tr, params)

        score_tr = knn_scores_from_distances(
            D_tr, y_tr, k=DISTANCE_K, weighted=model_cfg["weighted"], exclude_self=True
        )
        score_te = knn_scores_from_distances(
            D_te, y_tr, k=DISTANCE_K, weighted=model_cfg["weighted"], exclude_self=False
        )

    elif model_cfg["model_name"] == "centroid":
        score_tr = centroid_scores_from_distances(X_tr, y_tr, X_tr, params)
        score_te = centroid_scores_from_distances(X_tr, y_tr, X_te, params)

    else:
        raise ValueError(f"Unsupported model: {model_cfg['model_name']}")

    metrics = fold_metrics(y_te, score_te, y_tr, score_tr)

    out = {
        "family": "Distance-based functional",
        "model_name": model_cfg["model_name"],
        "distance": model_cfg["distance"],
        "method": f"{model_cfg['model_name']} + {model_cfg['distance']}",
    }
    out.update(metrics)
    return out


   ## 6. Rerun distance-based functional models

In [ ]:
distance_records = []

for sname in split_names:
    print(f"Current fold: {sname}")
    tr_idx = np.array(splits[sname]["train"], dtype=int)
    te_idx = np.array(splits[sname]["test"], dtype=int)

    for cfg in DISTANCE_MODELS:
        print(f"  -> {cfg['model_name']} + {cfg['distance']}")
        rec = run_distance_model_one_split(X, y, tr_idx, te_idx, cfg)
        rec["split"] = sname
        distance_records.append(rec)

df_distance_fold = pd.DataFrame(distance_records)
df_distance_summary = summarise_run(
    df_distance_fold,
    ["family", "model_name", "distance", "method"],
)

print("Distance fold metrics:", df_distance_fold.shape)
print("Distance summary:", df_distance_summary.shape)
display(df_distance_summary.sort_values(["f1_mean", "pr_auc_mean"], ascending=False).head(10))


Current fold: rep0_fold0
  -> weighted kNN + euclidean
  -> weighted kNN + cosine
  -> weighted kNN + seuclidean
  -> weighted kNN + correlation
  -> centroid + euclidean
  -> centroid + cosine
  -> centroid + seuclidean
  -> centroid + correlation
Current fold: rep0_fold1
  -> weighted kNN + euclidean
  -> weighted kNN + cosine
  -> weighted kNN + seuclidean
  -> weighted kNN + correlation
  -> centroid + euclidean
  -> centroid + cosine
  -> centroid + seuclidean
  -> centroid + correlation
Current fold: rep0_fold2
  -> weighted kNN + euclidean
  -> weighted kNN + cosine
  -> weighted kNN + seuclidean
  -> weighted kNN + correlation
  -> centroid + euclidean
  -> centroid + cosine
  -> centroid + seuclidean
  -> centroid + correlation
Current fold: rep0_fold3
  -> weighted kNN + euclidean
  -> weighted kNN + cosine
  -> weighted kNN + seuclidean
  -> weighted kNN + correlation
  -> centroid + euclidean
  -> centroid + cosine
  -> centroid + seuclidean
  -> centroid + correlation
Curr

,family,model_name,distance,method,pr_auc_mean,pr_auc_std,roc_auc_mean,roc_auc_std,sensitivity_mean,sensitivity_std,...,specificity_mean,specificity_std,accuracy_mean,accuracy_std,f1_mean,f1_std,youden_j_mean,youden_j_std,youden_threshold_mean,youden_threshold_std
7,Distance-based functional,weighted kNN,seuclidean,weighted kNN + seuclidean,0.808304,0.035406,0.891048,0.020077,0.776499,0.040170,...,0.942927,0.014496,0.909947,0.011811,0.773768,0.027378,0.719426,0.038035,0.282111,0.076271
6,Distance-based functional,weighted kNN,euclidean,weighted kNN + euclidean,0.736810,0.040613,0.854622,0.024432,0.685487,0.041446,...,0.931499,0.016621,0.882735,0.014043,0.698724,0.031923,0.616987,0.041082,0.305829,0.075045
5,Distance-based functional,weighted kNN,cosine,weighted kNN + cosine,0.736709,0.040452,0.854638,0.024412,0.685845,0.042392,...,0.929462,0.018436,0.881172,0.014569,0.696108,0.031373,0.615307,0.040626,0.313367,0.059862
3,Distance-based functional,centroid,seuclidean,centroid + seuclidean,0.573751,0.041169,0.799785,0.024732,0.734780,0.037229,...,0.768679,0.024407,0.761954,0.020622,0.550740,0.028079,0.503458,0.043671,0.337186,0.008689
1,Distance-based functional,centroid,cosine,centroid + cosine,0.539304,0.043801,0.777023,0.025532,0.720800,0.036209,...,0.727871,0.025907,0.726465,0.022095,0.511395,0.027299,0.448671,0.044905,0.310854,0.007598
2,Distance-based functional,centroid,euclidean,centroid + euclidean,0.540099,0.043804,0.777215,0.025559,0.719723,0.036629,...,0.727871,0.026036,0.726252,0.022256,0.510830,0.027600,0.447594,0.045437,0.354070,0.013206
4,Distance-based functional,weighted kNN,correlation,weighted kNN + correlation,0.545041,0.043815,0.728922,0.024417,0.484649,0.060223,...,0.886980,0.045307,0.807211,0.031104,0.499961,0.041552,0.371629,0.050565,0.251859,0.044097
0,Distance-based functional,centroid,correlation,centroid + correlation,0.372718,0.044441,0.677382,0.031218,0.709334,0.046406,...,0.542851,0.041612,0.575844,0.029476,0.398912,0.019379,0.252185,0.042450,0.399196,0.030392


   ## 7. Functional linear model helpers

In [ ]:
def mean_inner_cv_roc_auc_at_best_c(clf: LogisticRegressionCV) -> float:
    sc = clf.scores_
    if sc is None:
        return np.nan

    if isinstance(sc, dict):
        arr = np.asarray(next(iter(sc.values())))
    elif hasattr(sc, "ndim") and sc.ndim == 3:
        arr = sc[0]
    else:
        arr = np.asarray(sc)

    c_grid = np.asarray(clf.Cs_)
    best_c = float(clf.C_[0])
    j = int(np.argmin(np.abs(c_grid - best_c)))
    return float(np.mean(arr[:, j]))


def run_linear_model_one_split(
    X: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    kind: str,
) -> tuple[dict, np.ndarray]:
    X_tr = X[train_idx]
    X_te = X[test_idx]
    y_tr = y[train_idx]
    y_te = y[test_idx]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    if kind == "logreg_l2":
        model = LogisticRegressionCV(
            Cs=CS_GRID,
            cv=StratifiedKFold(n_splits=INNER_CV, shuffle=True, random_state=RANDOM_STATE),
            penalty="l2",
            solver="lbfgs",
            class_weight="balanced",
            scoring="roc_auc",
            max_iter=2000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        model.fit(X_tr_s, y_tr)
        score_tr = model.predict_proba(X_tr_s)[:, 1]
        score_te = model.predict_proba(X_te_s)[:, 1]
        coef = model.coef_[0].copy()
        extra = {
            "best_C": float(model.C_[0]),
            "best_cv_roc_auc": mean_inner_cv_roc_auc_at_best_c(model),
            "n_nonzero_coefs": int(X.shape[1]),
        }

    elif kind == "logreg_l1":
        model = LogisticRegressionCV(
            Cs=CS_GRID,
            cv=StratifiedKFold(n_splits=INNER_CV, shuffle=True, random_state=RANDOM_STATE),
            penalty="l1",
            solver="liblinear",
            class_weight="balanced",
            scoring="roc_auc",
            max_iter=4000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
        model.fit(X_tr_s, y_tr)
        score_tr = model.predict_proba(X_tr_s)[:, 1]
        score_te = model.predict_proba(X_te_s)[:, 1]
        coef = model.coef_[0].copy()
        extra = {
            "best_C": float(model.C_[0]),
            "best_cv_roc_auc": mean_inner_cv_roc_auc_at_best_c(model),
            "n_nonzero_coefs": int(np.sum(np.abs(coef) > 1e-6)),
        }

    elif kind == "linear_svm":
        inner_cv = StratifiedKFold(n_splits=INNER_CV, shuffle=True, random_state=RANDOM_STATE)
        c_scores = []

        for C in CS_GRID:
            fold_scores = []
            for inner_tr, inner_va in inner_cv.split(X_tr_s, y_tr):
                clf = LinearSVC(
                    C=C,
                    class_weight="balanced",
                    max_iter=10000,
                    random_state=RANDOM_STATE,
                )
                clf.fit(X_tr_s[inner_tr], y_tr[inner_tr])
                va_score = clf.decision_function(X_tr_s[inner_va])
                try:
                    auc = roc_auc_score(y_tr[inner_va], va_score)
                except ValueError:
                    auc = np.nan
                fold_scores.append(auc)

            c_scores.append(np.nanmean(fold_scores))

        best_i = int(np.nanargmax(c_scores))
        best_c = float(CS_GRID[best_i])

        model = LinearSVC(
            C=best_c,
            class_weight="balanced",
            max_iter=10000,
            random_state=RANDOM_STATE,
        )
        model.fit(X_tr_s, y_tr)
        score_tr = model.decision_function(X_tr_s)
        score_te = model.decision_function(X_te_s)
        coef = model.coef_[0].copy()
        extra = {
            "best_C": best_c,
            "best_cv_roc_auc": float(c_scores[best_i]),
            "n_nonzero_coefs": int(np.sum(np.abs(coef) > 1e-12)),
        }

    else:
        raise ValueError(f"Unsupported kind: {kind}")

    metrics = fold_metrics(y_te, score_te, y_tr, score_tr)
    metrics.update(extra)
    return metrics, coef


   ## 8. Rerun functional logistic regression and linear SVM

In [ ]:
linear_records = []
coef_store = {cfg["method"]: [] for cfg in LINEAR_MODELS}

for sname in split_names:
    print(f"Current fold: {sname}")
    tr_idx = np.array(splits[sname]["train"], dtype=int)
    te_idx = np.array(splits[sname]["test"], dtype=int)

    for cfg in LINEAR_MODELS:
        print(f"  -> {cfg['method']}")
        met, coef = run_linear_model_one_split(X, y, tr_idx, te_idx, cfg["kind"])

        row = {
            "split": sname,
            "family": cfg["family"],
            "method": cfg["method"],
        }
        row.update(met)
        linear_records.append(row)
        coef_store[cfg["method"]].append(coef)

df_linear_fold = pd.DataFrame(linear_records)
df_linear_summary = summarise_run(
    df_linear_fold,
    ["family", "method"],
)

print("\nLinear model processing complete.")
print("Linear fold metrics:", df_linear_fold.shape)
print("Linear summary:", df_linear_summary.shape)
display(df_linear_summary.sort_values(["f1_mean", "pr_auc_mean"], ascending=False))


Current fold: rep0_fold0
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep0_fold1
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep0_fold2
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep0_fold3
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep0_fold4
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep1_fold0
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep1_fold1
  -> Functional logistic regression (L2)
  -> Functional logistic regression (L1)
  -> Functional linear SVM
Current fold: rep1_fold2
  -> Functional logistic regression (

KeyboardInterrupt: 

   ## 9. Import FPCA / FPLS summary

In [ ]:
df_fpca_fpls = None

if FPCA_FPLS_SUMMARY_FILE.exists():
    df_fpca_fpls = pd.read_csv(FPCA_FPLS_SUMMARY_FILE).copy()
    print("Loaded FPCA/FPLS summary:", df_fpca_fpls.shape)
else:
    print("FPCA/FPLS summary not found. Final table will include only rerun models.")


   ## 10. Convert all sources into one final schema

In [ ]:
final_rows = []

for _, r in df_distance_summary.iterrows():
    final_rows.append({
        "source": "rerun",
        "family": r["family"],
        "method": r["method"],
        "representation": "L2-normalised sampled spectra",
        "tuning": f"k={DISTANCE_K}" if "kNN" in r["model_name"] else "class centroids",
        "pr_auc_mean": r["pr_auc_mean"],
        "pr_auc_std": r["pr_auc_std"],
        "roc_auc_mean": r["roc_auc_mean"],
        "roc_auc_std": r["roc_auc_std"],
        "sensitivity_mean": r["sensitivity_mean"],
        "sensitivity_std": r["sensitivity_std"],
        "precision_mean": r["precision_mean"],
        "precision_std": r["precision_std"],
        "specificity_mean": r["specificity_mean"],
        "specificity_std": r["specificity_std"],
        "accuracy_mean": r["accuracy_mean"],
        "accuracy_std": r["accuracy_std"],
        "f1_mean": r["f1_mean"],
        "f1_std": r["f1_std"],
        "youden_j_mean": r["youden_j_mean"],
        "youden_j_std": r["youden_j_std"],
    })

for _, r in df_linear_summary.iterrows():
    sub = df_linear_fold[df_linear_fold["method"] == r["method"]]
    best_c_mean = sub["best_C"].mean()
    tuning = f"inner-CV C | mean best C={best_c_mean:.4g}"

    final_rows.append({
        "source": "rerun",
        "family": r["family"],
        "method": r["method"],
        "representation": "L2-normalised sampled spectra",
        "tuning": tuning,
        "pr_auc_mean": r["pr_auc_mean"],
        "pr_auc_std": r["pr_auc_std"],
        "roc_auc_mean": r["roc_auc_mean"],
        "roc_auc_std": r["roc_auc_std"],
        "sensitivity_mean": r["sensitivity_mean"],
        "sensitivity_std": r["sensitivity_std"],
        "precision_mean": r["precision_mean"],
        "precision_std": r["precision_std"],
        "specificity_mean": r["specificity_mean"],
        "specificity_std": r["specificity_std"],
        "accuracy_mean": r["accuracy_mean"],
        "accuracy_std": r["accuracy_std"],
        "f1_mean": r["f1_mean"],
        "f1_std": r["f1_std"],
        "youden_j_mean": r["youden_j_mean"],
        "youden_j_std": r["youden_j_std"],
    })

if df_fpca_fpls is not None:
    df_use = df_fpca_fpls.copy()

    if "method_family" in df_use.columns and "classifier" in df_use.columns:
        df_use["family"] = "FPCA / FPLS"
        df_use["method"] = df_use["method_family"].astype(str) + " + " + df_use["classifier"].astype(str)
        df_use["representation"] = df_use["method_family"].astype(str)
        df_use["tuning"] = "J=" + df_use["J"].astype(str)

        for _, r in df_use.iterrows():
            final_rows.append({
                "source": "imported",
                "family": r["family"],
                "method": r["method"],
                "representation": r["representation"],
                "tuning": r["tuning"],
                "pr_auc_mean": r.get("pr_auc_mean", np.nan),
                "pr_auc_std": r.get("pr_auc_std", np.nan),
                "roc_auc_mean": r.get("roc_auc_mean", np.nan),
                "roc_auc_std": r.get("roc_auc_std", np.nan),
                "sensitivity_mean": r.get("sensitivity_mean", np.nan),
                "sensitivity_std": r.get("sensitivity_std", np.nan),
                "precision_mean": r.get("precision_mean", np.nan),
                "precision_std": r.get("precision_std", np.nan),
                "specificity_mean": r.get("specificity_mean", np.nan),
                "specificity_std": r.get("specificity_std", np.nan),
                "accuracy_mean": r.get("accuracy_mean", np.nan),
                "accuracy_std": r.get("accuracy_std", np.nan),
                "f1_mean": r.get("f1_mean", np.nan),
                "f1_std": r.get("f1_std", np.nan),
                "youden_j_mean": r.get("youden_j_mean", np.nan),
                "youden_j_std": r.get("youden_j_std", np.nan),
            })

final_df = pd.DataFrame(final_rows)
print("Final combined rows:", final_df.shape)


   ## 11. Final ranking tables

In [ ]:
rank_by_f1 = final_df.sort_values(
    ["f1_mean", "pr_auc_mean", "roc_auc_mean"],
    ascending=False,
    na_position="last",
).reset_index(drop=True)

print("\n=== TOP MODELS BY F1 ===")
display(
    rank_by_f1[
        ["source", "family", "method", "representation", "tuning", "f1_mean", "pr_auc_mean", "roc_auc_mean"]
    ].head(20)
)


   ## 12. Best method within each family

In [ ]:
best_by_family = (
    final_df.sort_values(["f1_mean", "pr_auc_mean"], ascending=False, na_position="last")
    .groupby("family", as_index=False)
    .first()
)

best_by_family_pretty = pd.DataFrame({
    "Family": best_by_family["family"],
    "Method": best_by_family["method"],
    "Representation": best_by_family["representation"],
    "Tuning": best_by_family["tuning"],
    "PR-AUC mean ± std": [ms(m, s) for m, s in zip(best_by_family["pr_auc_mean"], best_by_family["pr_auc_std"])],
    "ROC-AUC mean ± std": [ms(m, s) for m, s in zip(best_by_family["roc_auc_mean"], best_by_family["roc_auc_std"])],
    "F1 mean ± std": [ms(m, s) for m, s in zip(best_by_family["f1_mean"], best_by_family["f1_std"])],
})

print("\n=== BEST BY FAMILY ===")
display(best_by_family_pretty)


   ## 13. Build interpretability payload for the interpretability notebook

In [ ]:
mean_0 = X[y == 0].mean(axis=0)
mean_1 = X[y == 1].mean(axis=0)
diff_10 = mean_1 - mean_0

pca = PCA(n_components=2, random_state=RANDOM_STATE)
scores_fpca = pca.fit_transform(X)

pls = PLSRegression(n_components=2)
scores_fpls, _ = pls.fit_transform(X, y.astype(float).reshape(-1, 1))

first_split = split_names[0]
tr_idx = np.array(splits[first_split]["train"], dtype=int)
te_idx = np.array(splits[first_split]["test"], dtype=int)

X_tr = X[tr_idx]
X_te = X[te_idx]
y_tr = y[tr_idx]
y_te = y[te_idx]

test_pos = np.where(y_te == 1)[0]
example_te_local = int(test_pos[0]) if len(test_pos) else 0
x_query = X_te[example_te_local:example_te_local + 1]

D = pairwise_distances(x_query, X_tr, metric="euclidean").ravel()
nn_idx = np.argsort(D)[:DISTANCE_K]
closest_idx = int(nn_idx[0])
closest_curve = X_tr[closest_idx]
abs_diff = np.abs(x_query.ravel() - closest_curve)

scaler_full = StandardScaler()
X_full_s = scaler_full.fit_transform(X)

logreg_l2_best_c = float(
    df_linear_fold.loc[
        df_linear_fold["method"] == "Functional logistic regression (L2)", "best_C"
    ].median()
)

svm_best_c = float(
    df_linear_fold.loc[
        df_linear_fold["method"] == "Functional linear SVM", "best_C"
    ].median()
)

logreg_full = LogisticRegressionCV(
    Cs=[logreg_l2_best_c],
    cv=3,
    penalty="l2",
    solver="lbfgs",
    class_weight="balanced",
    scoring="roc_auc",
    max_iter=4000,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
logreg_full.fit(X_full_s, y)
beta_logreg = logreg_full.coef_[0].copy()

svm_full = LinearSVC(
    C=svm_best_c,
    class_weight="balanced",
    max_iter=10000,
    random_state=RANDOM_STATE,
)
svm_full.fit(X_full_s, y)
beta_svm = svm_full.coef_[0].copy()

interpretability_payload_path = OUT_DIR / "interpretability_payload.npz"

np.savez(
    interpretability_payload_path,
    wavelengths=wavelengths,
    y=y,
    mean_0=mean_0,
    mean_1=mean_1,
    diff_10=diff_10,
    scores_fpca=scores_fpca,
    scores_fpls=scores_fpls,
    x_query=x_query.ravel(),
    closest_curve=closest_curve,
    abs_diff=abs_diff,
    beta_logreg=beta_logreg,
    beta_svm=beta_svm,
)

print("Interpretability payload saved:", interpretability_payload_path)


   ## 14. Save main outputs

In [ ]:
distance_fold_path = OUT_DIR / "final_distance_fold_metrics.csv"
linear_fold_path = OUT_DIR / "final_linear_fold_metrics.csv"
final_summary_path = OUT_DIR / "final_model_summary.csv"
rank_f1_path = OUT_DIR / "final_rank_by_f1.csv"
best_family_path = OUT_DIR / "final_best_by_family.csv"

df_distance_fold.to_csv(distance_fold_path, index=False)
df_linear_fold.to_csv(linear_fold_path, index=False)
final_df.to_csv(final_summary_path, index=False)
rank_by_f1.to_csv(rank_f1_path, index=False)
best_by_family_pretty.to_csv(best_family_path, index=False)

print("Saved:")
print("-", distance_fold_path)
print("-", linear_fold_path)
print("-", final_summary_path)
print("-", rank_f1_path)
print("-", best_family_path)
print("-", interpretability_payload_path)


   ## 15. Short final view

In [ ]:
print("\n=== FINAL TOP 15 BY F1 ===")
display(
    rank_by_f1[
        ["family", "method", "representation", "tuning", "f1_mean", "pr_auc_mean", "roc_auc_mean"]
    ].head(15)
)

print("\n=== BEST BY FAMILY ===")
display(best_by_family_pretty)